[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajitpanday80/ai-learn/blob/main/notebooks/llm-how-built.ipynb)

# How are LLMs built?

Large Language Models (like GPT, Claude, and Llama) are built through a pipeline of distinct stages. In this notebook we'll walk through the core ideas hands-on using a small, real model (GPT-2) that runs comfortably on CPU or GPU in Colab.

**The pipeline, at a glance:**
1. **Data collection** — scrape/curate massive amounts of text (web pages, books, code, etc.)
2. **Tokenization** — break text into subword units the model can consume
3. **Architecture** — stack transformer decoder blocks (attention + feed-forward layers)
4. **Pretraining** — train the model to predict the next token, over and over, on trillions of tokens
5. **Fine-tuning / alignment (RLHF, instruction tuning)** — shape the raw "next-token predictor" into a helpful assistant

We'll inspect each of these with real code below. GPT-2 is small enough to run on CPU, but if you want it to feel snappier, go to **Runtime > Change runtime type > GPU**.

In [ ]:
!pip install -q transformers torch matplotlib

import torch
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('dark_background')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cpu':
    print("Tip: Runtime > Change runtime type > GPU for a speed boost (not required for this notebook).")

## Stage 1: Tokenization

LLMs don't see raw text — they see sequences of integers called **tokens**. Most modern LLMs use **byte-pair encoding (BPE)**, which learns a vocabulary of common subwords from the training corpus. Common words become a single token; rare words get split into pieces.

Let's load GPT-2's tokenizer and see this in action.

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# EXPERIMENT: change this text and see how it gets split into tokens!
text = "Large language models are built by predicting the next token, over and over again."

token_ids = tokenizer.encode(text)
tokens = [tokenizer.decode([t]) for t in token_ids]

print(f"Vocabulary size: {tokenizer.vocab_size:,} tokens")
print(f"Text split into {len(tokens)} tokens:\n")
for tid, tok in zip(token_ids, tokens):
    print(f"  {tid:>6}  ->  {tok!r}")

## Stage 2 & 3: Architecture + Pretraining Objective

The architecture behind almost every modern LLM is the **transformer decoder**: a stack of blocks, each containing:
- **Self-attention** — lets each token "look at" every previous token to gather context
- **Feed-forward layers** — process that context further
- **Residual connections + layer norm** — help gradients flow through very deep stacks

During **pretraining**, the model is given a huge amount of text and trained on one deceptively simple objective: **predict the next token**, given all previous tokens. Repeated over trillions of tokens, this single objective is enough to teach a model grammar, facts, reasoning patterns, and style.

Let's load the actual pretrained GPT-2 model, inspect its architecture, and watch it predict next-token probabilities.

In [ ]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
n_layers = model.config.n_layer
n_heads = model.config.n_head
d_model = model.config.n_embd

print(f"Total parameters: {n_params:,}")
print(f"Transformer decoder blocks (layers): {n_layers}")
print(f"Attention heads per block: {n_heads}")
print(f"Hidden dimension (d_model): {d_model}")

In [ ]:
# EXPERIMENT: change the prompt and see what the model thinks comes next!
prompt = "The capital of France is"

input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    logits = model(input_ids).logits

next_token_logits = logits[0, -1, :]
probs = torch.softmax(next_token_logits, dim=-1)

top_k = 10
top_probs, top_ids = torch.topk(probs, top_k)
top_tokens = [tokenizer.decode([tid]) for tid in top_ids]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(range(top_k), top_probs.cpu().numpy(), color='#4fd1c5')
ax.set_xticks(range(top_k))
ax.set_xticklabels(top_tokens, rotation=45, ha='right')
ax.set_ylabel('Probability')
ax.set_title(f'Next-token predictions for: "{prompt}"')
plt.tight_layout()
plt.show()

print(f"\nThis is literally the pretraining objective in action: given '{prompt}',")
print("the model outputs a probability distribution over every possible next token.")

## Stage 4: From next-token predictor to helpful assistant

A raw pretrained model is just an extremely good autocomplete — it will happily continue text in whatever style it started (including unhelpful or unsafe directions). To turn it into an assistant like ChatGPT or Claude, builders add:

- **Instruction / supervised fine-tuning (SFT)** — train further on (prompt, ideal response) pairs written by humans
- **RLHF (Reinforcement Learning from Human Feedback)** — humans rank multiple model outputs; a reward model learns their preferences; the LLM is optimized against that reward model
- **Safety tuning** — additional training to refuse harmful requests and reduce hallucinations

We can approximate one piece of this intuition — **sampling strategy** — since even at inference time, *how* you sample from the next-token distribution changes the model's behavior a lot. Low **temperature** = safe/predictable; high temperature = creative/random.

In [ ]:
# EXPERIMENT: try different temperatures (0.1 = very predictable, 1.5 = very random)
prompt = "Once upon a time, the AI model"
temperatures = [0.2, 0.7, 1.5]

input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

for temp in temperatures:
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=25,
            do_sample=True,
            temperature=temp,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"--- temperature={temp} ---")
    print(generated)
    print()